# 03 - Build spatial structure and exposure definitions

This notebook creates the spatial backbone for later analysis. It does not estimate spillover effects yet.

It does five things:

1. loads the treatment panel and matched grid geometry,
2. computes representative grid-cell locations,
3. builds candidate spatial weights objects,
4. defines a catalog of candidate exposure mappings,
5. saves edge lists, summaries, and diagnostic plots for later notebooks.

The default settings are intentionally conservative so the notebook runs comfortably on the current grid. You can expand the radius and ring supports later from the configuration cell.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

def detect_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for p in candidates:
        if (p / "notebooks").exists():
            return p
    return cwd

PROJECT_ROOT = detect_project_root()
project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.spatial_structure_utils import (
    InverseDistanceSpec,
    add_inverse_distance_weights,
    build_exposure_mapping_catalog,
    build_grid_centroids,
    build_knn_neighbor_edges,
    build_radius_neighbor_edges,
    build_ring_neighbor_edges,
    compute_neighbor_exposure,
    summarize_weight_structure,
)

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True

DATA_DIR = PROJECT_ROOT / "data"
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

for p in [DATA_DIR, INTERMEDIATE_DIR, OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## User configuration

Edit the candidate supports here if you want a denser or sparser spatial structure. The default configuration creates two binary radius supports, three k-nearest-neighbor supports, two inverse-distance supports on top of kNN graphs, and two ring supports.

In [ ]:
PANEL_TREATMENT_PATH = None
GRID_INPUT_PATH = None

PANEL_CELL_ID_COL = "cell_id"
PANEL_YEAR_COL = "year"
GRID_CELL_ID_COL = "cell_id"

AREA_CRS = "EPSG:6933"
RADIUS_VALUES_KM = [25, 50]
K_VALUES = [4, 8, 16]
INVERSE_DISTANCE_SPECS = [
    {"support_name": "knn_k08", "power": 1.0, "min_distance_m": 100.0, "row_standardize": True},
    {"support_name": "knn_k16", "power": 1.0, "min_distance_m": 100.0, "row_standardize": True},
]
RING_BOUNDS_KM = [(0, 25), (25, 50)]

TREATMENT_KEYS = ["protected_area", "afro", "indigena", "campesino", "land_title_any"]
PREVIEW_TREATMENT_KEY = "protected_area"
PREVIEW_SUPPORT_NAME = "radius_50km"

SPATIAL_DIR = INTERMEDIATE_DIR / "spatial_structure"
SPATIAL_WEIGHT_SUMMARY_TABLE_NAME = "03_spatial_weight_summary.csv"
EXPOSURE_CATALOG_TABLE_NAME = "03_exposure_mapping_catalog.csv"
SPATIAL_SUMMARY_JSON_NAME = "03_spatial_structure_summary.json"
NEIGHBOR_COUNT_FIG_NAME = "03_neighbor_count_distributions.png"
DISTANCE_FIG_NAME = "03_distance_distributions.png"


## Helper functions

These helpers keep the notebook self-contained and aligned with the conventions used in notebooks 01 and 02.

In [ ]:
def ensure_exists(path_like, label):
    path = Path(path_like)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path

def resolve_latest_input(explicit_path, glob_pattern):
    if explicit_path is not None:
        return ensure_exists(explicit_path, "Input file")
    candidates = sorted(INTERMEDIATE_DIR.glob(glob_pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No files matched {glob_pattern!r} in {INTERMEDIATE_DIR}")
    return candidates[0]

def infer_run_tag_from_input(path_like):
    stem = Path(path_like).stem
    for prefix in ["panel_treatment", "panel_raw_long", "cell_treatment_lookup", "grid_geometry"]:
        if stem.startswith(prefix):
            return stem[len(prefix):].lstrip("_")
    return ""

def append_tag_to_filename(filename, tag):
    path = Path(filename)
    suffix = "".join(path.suffixes)
    stem = path.name[:-len(suffix)] if suffix else path.name
    return f"{stem}_{tag}{suffix}" if tag else path.name

def choose_best_grid_path(panel_df, explicit_grid_path=None, grid_pattern="grid_geometry*.geojson"):
    panel_ids = set(panel_df[PANEL_CELL_ID_COL].astype("string").unique())
    if explicit_grid_path is not None:
        candidates = [ensure_exists(explicit_grid_path, "Grid geometry file")]
    else:
        candidates = sorted(INTERMEDIATE_DIR.glob(grid_pattern))
    if not candidates:
        raise FileNotFoundError(f"No grid geometry files matched {grid_pattern!r} in {INTERMEDIATE_DIR}")

    scored = []
    for candidate in candidates:
        gdf = gpd.read_file(candidate)
        if GRID_CELL_ID_COL not in gdf.columns:
            continue
        grid_ids = set(gdf[GRID_CELL_ID_COL].astype("string").unique())
        shared = len(panel_ids & grid_ids)
        scored.append((shared, -abs(len(grid_ids) - len(panel_ids)), candidate, gdf))
    if not scored:
        raise ValueError("No candidate grid file contained the expected grid cell id column.")
    scored.sort(reverse=True, key=lambda item: (item[0], item[1]))
    best_shared, _, best_path, best_gdf = scored[0]
    if best_shared == 0:
        raise ValueError("No overlap was found between the panel cell ids and the candidate grid files.")
    return best_path, best_gdf

def save_fig(path_like):
    path = Path(path_like)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved figure:", path)

def try_write_json(payload, path_like):
    path = Path(path_like)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print("Saved:", path)

def strip_frame(ax):
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

def support_slug(value):
    return str(value).replace(".", "p").replace("-", "m")


## Load the treated panel and compute representative grid locations

This block reads the panel saved by notebook 02, matches the grid geometry, filters the grid to panel cells, and computes projected centroids to use in the spatial queries.

In [ ]:
panel_input_path = resolve_latest_input(PANEL_TREATMENT_PATH, "panel_treatment*.parquet")
panel_df = pd.read_parquet(panel_input_path)
required_panel_cols = [PANEL_CELL_ID_COL, PANEL_YEAR_COL]
missing_panel_cols = [c for c in required_panel_cols if c not in panel_df.columns]
if missing_panel_cols:
    raise ValueError(f"Missing required columns in the panel file: {missing_panel_cols}")

panel_df[PANEL_CELL_ID_COL] = panel_df[PANEL_CELL_ID_COL].astype("string")
panel_df[PANEL_YEAR_COL] = pd.to_numeric(panel_df[PANEL_YEAR_COL], errors="coerce").astype("Int64")
if panel_df[PANEL_YEAR_COL].isna().any():
    raise ValueError("The panel year column contains invalid or missing values.")

available_treatment_keys = [key for key in TREATMENT_KEYS if f"treated_it_{key}" in panel_df.columns]
if not available_treatment_keys:
    raise ValueError("The panel does not contain any treatment indicators from notebook 02.")
if PREVIEW_TREATMENT_KEY not in available_treatment_keys:
    PREVIEW_TREATMENT_KEY = available_treatment_keys[0]

run_tag = infer_run_tag_from_input(panel_input_path)
grid_path, grid_gdf = choose_best_grid_path(panel_df, GRID_INPUT_PATH)
if not run_tag:
    run_tag = infer_run_tag_from_input(grid_path)
grid_gdf[GRID_CELL_ID_COL] = grid_gdf[GRID_CELL_ID_COL].astype("string")
grid_gdf = grid_gdf.drop_duplicates(subset=[GRID_CELL_ID_COL]).copy()
panel_ids = set(panel_df[PANEL_CELL_ID_COL].unique())
grid_ids = set(grid_gdf[GRID_CELL_ID_COL].unique())
missing_grid_ids = sorted(panel_ids - grid_ids)
grid_gdf = grid_gdf[grid_gdf[GRID_CELL_ID_COL].isin(panel_ids)].copy()
if missing_grid_ids:
    raise ValueError("The selected grid geometry does not cover every panel cell. Missing ids include: " + str(missing_grid_ids[:10]))

centroid_gdf = build_grid_centroids(grid_gdf, grid_cell_id_col=GRID_CELL_ID_COL, area_crs=AREA_CRS)

print("Panel input:", panel_input_path)
print("Grid geometry:", grid_path)
print("Run tag:", run_tag if run_tag else "<none>")
print("Panel rows:", panel_df.shape[0])
print("Panel cells:", panel_df[PANEL_CELL_ID_COL].nunique())
print("Year range:", int(panel_df[PANEL_YEAR_COL].min()), "-", int(panel_df[PANEL_YEAR_COL].max()))
print("Available treatment keys:", ", ".join(available_treatment_keys))
print(centroid_gdf[[GRID_CELL_ID_COL, "centroid_lon", "centroid_lat"]].head(5).to_string(index=False))


## Build candidate spatial weights and define exposure mappings

The notebook saves portable weight objects as edge lists rather than opaque Python pickles. Later notebooks can read these Parquet files and use the catalog to decide which exposure mapping to estimate.

In [ ]:
SPATIAL_DIR.mkdir(parents=True, exist_ok=True)
all_cell_ids = centroid_gdf[GRID_CELL_ID_COL].astype("string")

radius_edges = {}
for radius_km in RADIUS_VALUES_KM:
    support_name = f"radius_{int(radius_km)}km"
    radius_edges[support_name] = build_radius_neighbor_edges(
        centroid_gdf,
        radius_m=float(radius_km) * 1000.0,
        grid_cell_id_col=GRID_CELL_ID_COL,
        support_name=support_name,
    )

knn_edges = {}
for k in K_VALUES:
    support_name = f"knn_k{int(k):02d}"
    knn_edges[support_name] = build_knn_neighbor_edges(
        centroid_gdf,
        k=int(k),
        grid_cell_id_col=GRID_CELL_ID_COL,
        support_name=support_name,
    )

inverse_distance_specs = [InverseDistanceSpec(**spec) for spec in INVERSE_DISTANCE_SPECS]
invdist_edges = {}
for spec in inverse_distance_specs:
    if spec.support_name not in knn_edges:
        raise ValueError(f"Inverse-distance support {spec.support_name!r} is not available in the kNN support set.")
    invdist_name = f"invdist_{spec.support_name}_p{support_slug(spec.power)}"
    invdist_edges[invdist_name] = add_inverse_distance_weights(
        knn_edges[spec.support_name],
        power=spec.power,
        min_distance_m=spec.min_distance_m,
        row_standardize=spec.row_standardize,
        support_name=invdist_name,
    )

ring_edges = {}
for inner_km, outer_km in RING_BOUNDS_KM:
    support_name = f"ring_{int(inner_km)}_{int(outer_km)}km"
    ring_edges[support_name] = build_ring_neighbor_edges(
        centroid_gdf,
        inner_radius_m=float(inner_km) * 1000.0,
        outer_radius_m=float(outer_km) * 1000.0,
        grid_cell_id_col=GRID_CELL_ID_COL,
        support_name=support_name,
    )

weight_objects = {}
weight_objects.update(radius_edges)
weight_objects.update(knn_edges)
weight_objects.update(invdist_edges)
weight_objects.update(ring_edges)

weight_summary_rows = [
    summarize_weight_structure(edges_df, all_cell_ids=all_cell_ids, support_name=support_name)
    for support_name, edges_df in weight_objects.items()
]
weight_summary_df = pd.DataFrame(weight_summary_rows).sort_values(["support_type", "support_name"]).reset_index(drop=True)

exposure_catalog_df = build_exposure_mapping_catalog(
    radius_values_km=[float(v) for v in RADIUS_VALUES_KM],
    k_values=[int(v) for v in K_VALUES],
    inverse_distance_specs=inverse_distance_specs,
    ring_bounds_km=[(float(inner), float(outer)) for inner, outer in RING_BOUNDS_KM],
)

centroid_path = SPATIAL_DIR / append_tag_to_filename("03_grid_centroids.parquet", run_tag)
centroid_gdf.to_parquet(centroid_path, index=False)
weight_file_map = {}
for support_name, edges_df in weight_objects.items():
    edge_path = SPATIAL_DIR / append_tag_to_filename(f"03_{support_name}_edges.parquet", run_tag)
    edges_df.to_parquet(edge_path, index=False)
    weight_file_map[support_name] = str(edge_path)

weight_summary_path = TABLE_DIR / append_tag_to_filename(SPATIAL_WEIGHT_SUMMARY_TABLE_NAME, run_tag)
exposure_catalog_path = TABLE_DIR / append_tag_to_filename(EXPOSURE_CATALOG_TABLE_NAME, run_tag)
summary_json_path = TABLE_DIR / append_tag_to_filename(SPATIAL_SUMMARY_JSON_NAME, run_tag)
weight_summary_df.to_csv(weight_summary_path, index=False)
exposure_catalog_df.to_csv(exposure_catalog_path, index=False)

spatial_summary = {
    "panel_input_path": str(panel_input_path),
    "grid_input_path": str(grid_path),
    "centroid_path": str(centroid_path),
    "n_cells": int(centroid_gdf.shape[0]),
    "n_panel_rows": int(panel_df.shape[0]),
    "year_min": int(panel_df[PANEL_YEAR_COL].min()),
    "year_max": int(panel_df[PANEL_YEAR_COL].max()),
    "radius_values_km": [float(v) for v in RADIUS_VALUES_KM],
    "k_values": [int(v) for v in K_VALUES],
    "inverse_distance_specs": [
        {
            "support_name": spec.support_name,
            "power": float(spec.power),
            "min_distance_m": float(spec.min_distance_m),
            "row_standardize": bool(spec.row_standardize),
        }
        for spec in inverse_distance_specs
    ],
    "ring_bounds_km": [[float(inner), float(outer)] for inner, outer in RING_BOUNDS_KM],
    "weight_files": weight_file_map,
}
try_write_json(spatial_summary, summary_json_path)

preview_support_name = PREVIEW_SUPPORT_NAME if PREVIEW_SUPPORT_NAME in weight_objects else next(iter(weight_objects.keys()))
preview_treated_col = f"treated_it_{PREVIEW_TREATMENT_KEY}"
if preview_support_name in weight_objects and preview_treated_col in panel_df.columns:
    preview_year = int(panel_df[PANEL_YEAR_COL].max())
    preview_status = panel_df.loc[
        panel_df[PANEL_YEAR_COL] == preview_year,
        [PANEL_CELL_ID_COL, PANEL_YEAR_COL, preview_treated_col],
    ].copy()
    preview_any = compute_neighbor_exposure(
        preview_status,
        weight_objects[preview_support_name],
        cell_id_col=PANEL_CELL_ID_COL,
        year_col=PANEL_YEAR_COL,
        value_col=preview_treated_col,
        aggregation="max",
    ).rename(columns={"exposure": "exposure_any"})
    preview_share = compute_neighbor_exposure(
        preview_status,
        weight_objects[preview_support_name],
        cell_id_col=PANEL_CELL_ID_COL,
        year_col=PANEL_YEAR_COL,
        value_col=preview_treated_col,
        aggregation="mean",
    ).rename(columns={"exposure": "exposure_share"})
    preview_exposure_df = preview_any.merge(preview_share, on=[PANEL_CELL_ID_COL, PANEL_YEAR_COL], how="inner")
    preview_exposure_df = preview_exposure_df.sort_values(["exposure_any", "exposure_share"], ascending=[False, False]).head(10)
else:
    preview_exposure_df = pd.DataFrame()

print("Saved:")
print(" -", centroid_path)
print(" -", weight_summary_path)
print(" -", exposure_catalog_path)
print(" -", summary_json_path)
print(weight_summary_df.to_string(index=False))
print(exposure_catalog_df.head(12).to_string(index=False))
if not preview_exposure_df.empty:
    print(preview_exposure_df.to_string(index=False))


## Diagnostic plots

These plots help assess whether the candidate support sets are sensible before any exposure-based estimation. The first figure focuses on neighbors-per-cell distributions, while the second focuses on edge-distance distributions.

In [ ]:
neighbor_count_fig_path = FIG_DIR / append_tag_to_filename(NEIGHBOR_COUNT_FIG_NAME, run_tag)
distance_fig_path = FIG_DIR / append_tag_to_filename(DISTANCE_FIG_NAME, run_tag)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for support_name, edges_df in radius_edges.items():
    counts = edges_df.groupby("source_cell_id")["neighbor_cell_id"].size().reindex(all_cell_ids, fill_value=0)
    axes[0].hist(counts, bins=40, histtype="step", linewidth=1.6, label=support_name)
axes[0].set_title("Neighbors per cell: binary radius supports")
axes[0].set_xlabel("Neighbors per cell")
axes[0].set_ylabel("Grid cells")
axes[0].legend(frameon=False, fontsize=9)
axes[0].grid(axis="y", alpha=0.3)
axes[0].xaxis.grid(False)
axes[0].tick_params(axis="x", labelsize=9)
axes[0].tick_params(axis="y", labelsize=9)
strip_frame(axes[0])

for support_name, edges_df in ring_edges.items():
    counts = edges_df.groupby("source_cell_id")["neighbor_cell_id"].size().reindex(all_cell_ids, fill_value=0)
    axes[1].hist(counts, bins=40, histtype="step", linewidth=1.6, label=support_name)
axes[1].set_title("Neighbors per cell: ring supports")
axes[1].set_xlabel("Neighbors per cell")
axes[1].set_ylabel("Grid cells")
axes[1].legend(frameon=False, fontsize=9)
axes[1].grid(axis="y", alpha=0.3)
axes[1].xaxis.grid(False)
axes[1].tick_params(axis="x", labelsize=9)
axes[1].tick_params(axis="y", labelsize=9)
strip_frame(axes[1])
save_fig(neighbor_count_fig_path)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for support_name, edges_df in radius_edges.items():
    axes[0].hist(edges_df["distance_m"] / 1000.0, bins=50, histtype="step", linewidth=1.6, label=support_name)
axes[0].set_title("Edge-distance distributions: radius supports")
axes[0].set_xlabel("Distance (km)")
axes[0].set_ylabel("Directed edges")
axes[0].legend(frameon=False, fontsize=9)
axes[0].grid(axis="y", alpha=0.3)
axes[0].xaxis.grid(False)
axes[0].tick_params(axis="x", labelsize=9)
axes[0].tick_params(axis="y", labelsize=9)
strip_frame(axes[0])

for support_name, edges_df in knn_edges.items():
    k_rank = int(edges_df["neighbor_rank"].dropna().max()) if not edges_df.empty else 0
    kth_distances = edges_df.loc[edges_df["neighbor_rank"] == k_rank, "distance_m"] / 1000.0
    axes[1].hist(kth_distances, bins=50, histtype="step", linewidth=1.6, label=f"{support_name} (k-th)")
axes[1].set_title("Distance to the k-th nearest neighbor")
axes[1].set_xlabel("Distance (km)")
axes[1].set_ylabel("Grid cells")
axes[1].legend(frameon=False, fontsize=9)
axes[1].grid(axis="y", alpha=0.3)
axes[1].xaxis.grid(False)
axes[1].tick_params(axis="x", labelsize=9)
axes[1].tick_params(axis="y", labelsize=9)
strip_frame(axes[1])
save_fig(distance_fig_path)
plt.show()


## Final notes

The key outputs from this notebook are:

- centroid coordinates for each grid cell,
- candidate weight objects saved as Parquet edge lists,
- a catalog of candidate exposure mappings,
- summary tables for the support sets,
- neighbor-count and distance-distribution figures.

Later notebooks can choose one or more entries from the exposure catalog and merge the corresponding support set with the treatment panel to compute realized spillover exposures.